# S13 · A network that keeps the picture (a tiny CNN)

The network in the last notebook flattened each 8x8 image into a flat list of 64
numbers. In doing so it threw away the fact that some pixels sit next to each
other, above each other, in a grid. But a "7" is a "7" because of how its strokes
are arranged in space.

A **convolutional network** (CNN) keeps the grid. It slides tiny pattern-detectors
over the image to find local features like edges, the way you might recognise a
digit by the strokes that make it up. It is the family behind most modern image
recognition.

**New here? Read this once.**

- New to Python? You can still run every cell and read along. The training recipe
  is the same five steps as the last notebook.
- Curious how "seeing local patterns" works? Step 1 shows one filter by hand, no
  maths required.
- Already confident? The optional pretrained-model section at the very end points
  you at the real thing (it needs internet, so we only describe it).
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there. Colab already ships
PyTorch, scikit-learn, NumPy and matplotlib, so there is nothing to install.

In [ ]:
# Everything this notebook uses (torch, scikit-learn, numpy, matplotlib)
# already ships with Colab, so there is nothing to install here.
print("Setup complete - nothing to install.")

In [ ]:
import numpy as np                       # fast maths on lists of numbers
import matplotlib.pyplot as plt           # drawing charts
from sklearn.datasets import load_digits  # the handwritten-digits dataset
from sklearn.model_selection import train_test_split  # to hold out a test set

np.random.seed(0)

## Step 1 — what a convolution does, by hand

A **convolution** slides a small **filter** (a tiny grid of weights) over the
image. At each spot it lays the filter over a small patch of pixels, multiplies
them together, and adds them up into one number. The result is a **feature map**
that lights up wherever the filter's pattern appears.

- Early filters learn simple things like edges.
- Later layers combine edges into shapes.
- The filter's numbers are learned by the same rolling-downhill as before.

Let us watch one hand-made edge filter slide over a single digit.

In [ ]:
digits = load_digits()

# Take one digit image (an 8x8 grid of pixel values).
one_image = digits.images[0]

# A simple 3x3 filter that responds to vertical edges
# (dark on the left, bright on the right, or the other way round).
edge_filter = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1],
], dtype=float)

# Slide the filter over the image and record the response at each spot.
feature_map = np.zeros((6, 6))
for row in range(6):
    for col in range(6):
        patch = one_image[row:row + 3, col:col + 3]
        feature_map[row, col] = np.sum(patch * edge_filter)

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
axes[0].imshow(one_image, cmap="gray_r")
axes[0].set_title("input digit")
axes[1].imshow(edge_filter, cmap="coolwarm")
axes[1].set_title("a 3x3 edge filter")
axes[2].imshow(np.abs(feature_map), cmap="magma")
axes[2].set_title("feature map\n(lights up at edges)")
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

print("A convolution slides a small filter over the image to find a pattern.")

## Step 2 — load the digits and keep the 2D shape

A PyTorch convolution wants each image with a **channel** dimension, shaped
`(channels, height, width)`. Our digits are grey, so there is 1 channel:
`(1, 8, 8)`. We scale the pixels to 0 to 1 and, unlike last time, we do **not**
flatten them, so the grid survives.

In [ ]:
# Scale pixels to 0..1 and keep the 8x8 image shape (do NOT flatten this time).
images = digits.images / 16.0
labels = digits.target

# Add a channel dimension so each image is (1, 8, 8): one grey channel.
images = images.reshape(-1, 1, 8, 8)

# Split into training and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.2, random_state=0)

print("one training image shape:", X_train.shape[1:], "(channels, height, width)")
print("training examples       :", X_train.shape[0])
print("test examples           :", X_test.shape[0])

## Step 3 — make PyTorch tensors

The same conversion as the last notebook: inputs to float tensors, labels to long
(integer) tensors.

In [ ]:
import torch

# Set the seed so the random starting dials are the same each run.
torch.manual_seed(0)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

print("X_train tensor shape:", tuple(X_train_t.shape))

## Step 4 — build a small CNN

Our little convolutional network:

- **Conv2d**: 1 input channel → 8 filters, each 3x3, so it learns 8 patterns,
- **ReLU**: bend the feature maps,
- **Flatten**: lay the feature maps out into one long list,
- **Linear**: turn that list into 10 scores, one per digit.

With `padding=1` the 3x3 convolution keeps the image at 8x8, so after the
convolution we have 8 feature maps of 8x8 = 512 numbers feeding the final
layer.

In [ ]:
from torch import nn

# A small convolutional network.
model = nn.Sequential(
    # 1 input channel -> 8 filters, each 3x3. padding=1 keeps the size at 8x8.
    nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1),
    nn.ReLU(),
    # Flatten the 8 feature maps (8 * 8 * 8 = 512 numbers) into one list.
    nn.Flatten(),
    # Final linear layer: 512 numbers -> 10 digit scores.
    nn.Linear(8 * 8 * 8, 10),
)

print(model)

## Step 5 — how to measure mistakes, and how to nudge

Exactly as in the last notebook: cross-entropy loss for sorting into classes, and
the Adam optimiser to nudge the dials downhill.

In [ ]:
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

print("Loss and optimiser ready.")

## Step 6 — train for a few epochs

The same five-step loop as before: forward, loss, backward, nudge, repeat. A CNN
is trained in exactly the same way as the flat network; only the shape of the
layers changed.

In [ ]:
number_of_epochs = 20

loss_history = []

for epoch in range(number_of_epochs):
    # Forward pass: score each digit.
    predictions = model(X_train_t)

    # Loss: how wrong are the scores?
    loss = loss_function(predictions, y_train_t)

    # Backward pass and a downhill nudge.
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())

    if (epoch + 1) % 5 == 0:
        print("epoch", epoch + 1, " loss:", round(loss.item(), 4))

## Step 7 — watch the loss fall

The loss should drop as the CNN learns useful filters.

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(loss_history, color="#2E75B6")
plt.xlabel("epoch")
plt.ylabel("training loss")
plt.title("The CNN's loss falls as it learns its filters")
plt.show()

## Step 8 — the honest test

How often does the CNN read the right digit on images it never saw?

In [ ]:
with torch.no_grad():
    test_scores = model(X_test_t)

predicted_digits = torch.argmax(test_scores, dim=1)

number_correct = (predicted_digits == y_test_t).sum().item()
accuracy = number_correct / len(y_test_t)

print("correct predictions:", number_correct, "out of", len(y_test_t))
print("test accuracy      :", round(accuracy * 100, 2), "%")

## Optional, and needs internet — running a big pretrained network

*This section is a description only. There is no code cell to run, on purpose, so
the notebook always works offline. Try it later on Colab if you are curious.*

The CNN above is tiny and learns from scratch. In real projects people rarely
start from scratch. Instead they download a large network that a company already
trained on millions of photos, and reuse it. This is called using a **pretrained
model**, and it is how you can recognise a thousand kinds of object with almost no
training of your own.

If you were on Colab with internet, the sketch would be:

1. `pip install torchvision` (an image add-on for PyTorch that our offline course
   environment does not include).
2. Load a pretrained network, for example `torchvision.models.resnet18(weights="DEFAULT")`.
   This downloads the trained dials over the internet the first time.
3. Feed it a photo and read off its top guesses.

We deliberately stop here. It needs a download and an extra library, and it would
break an offline run, so it stays a description. The important news is that this
big model runs on the *same engine you built today*: linear maps, a bend between
them, a loss, and rolling downhill. It is only much larger, and trained on far
more pictures. Everything impressive you have heard of, image recognition and
beyond, is this foundation scaled up.

## What you just did

You trained a small convolutional network on the same digits. Instead of
flattening each image, the CNN slid learned filters over the 2D grid to find local
patterns like edges, then combined them to read the digit. Stacking many such
layers is how modern image models go from edges to shapes to whole objects. And
the training recipe, forward, loss, backprop, nudge, was identical to every other
network in this course.

That closes the machine-learning module. You have now met the three great model
families: straight-line models, trees, and neural networks. Next module: finance.